<center>


# **Shared Functions**

</center>

## **Purpose**

#### This is a compilation of utility functions designed for constructing the data platform. The programming principle of 'Single Responsibility' is adhered to; for instance, 'write_to_delta_overwrite' serves as a low-level function specifically crafted for overwrites having an ID policy in Delta Lake.

#### A more advanced use case, could be making this a python wheel and import the functions as a library

## **Imports**
#### Imports contains code snippets and type definitions for certain functionalities that will serve as the foundation for our upcoming development.

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
import json

## **Meta data functions**

#### We are constructing a metadata structure intended for subsequent entity-by-entity processing. The objective is to create a data structure that outlines the reading, processing instructions and ideel how to write for each table.

#### While the current example is simple/basic, it is expected to expand in complexity as the project grows.

In [ ]:
def _get_reader_dictionary(source: str, source_format: str):
    readers = {
        'parquet' : lambda : spark.read.parquet(source),
        'csv' : lambda : spark.read.options(delimiter=';', header=True).csv(source),
        'image' : lambda : (
            spark.read.format('binaryFile')
                 .option('pathGlobFilter', '*.png')
                 .option('recursiveFileLookup', 'false')
                 .load(source)
                 .withColumnRenamed('path', 'url')
        ),
    }
    return readers[source_format]

def _split_destination(destination: str) -> tuple:
    """
    Destination strings are always '<schema>.<table>' so writes against
    schema-enabled Fabric lakehouses can be fully qualified as
    '<lakehouse>.<schema>.<table>'.
    """
    if "." not in destination:
        raise ValueError(f"destination {destination!r} must be in the form '<schema>.<table>'")
    schema, table = destination.split(".", 1)
    return schema, table

def _process_meta_data(meta_entity: dict) -> dict:
    source = meta_entity["source"]
    destination = meta_entity["destination"]

    return {
        'source': source,
        'destination': destination,
        'reader_function': _get_reader_dictionary(source, meta_entity["format"]),
        'projection': meta_entity["projected_columns"],
        'custom_transform': meta_entity.get("custom_transform")
    }

def read_meta_from_json(path) -> dict:
    """
    Build a meta data structure describing how to process tables.

    Args:
        path (str): Path to the meta_data.json file.

    Returns:
        dict: Processed meta data keyed by destination '<schema>.<table>'.
    """
    json_file = open(path)
    raw_meta_data = json.load(json_file)

    processed_meta_data = {}
    for meta_entity in raw_meta_data:
        processed_meta_data[meta_entity['destination']] = _process_meta_data(meta_entity)

    return processed_meta_data


## **Write functions**

##### Functions for writing our data. Function 'write_to_delta_overwrite' is the low level function for overwriting data, the next for the reader would be adding low level functions for append, merge etc. What we have of polcies regarding id, audit etc can be added here, or on the next level. 

##### Next level functions, like load_dimension and load_fact, for handling higher level functionality, like preparing SCD2, policies like audit or other regarding loading fact and dimensions. In this case, load_dimension and load_fact are quite simple.

In [ ]:
def write_to_delta_overwrite(df: DataFrame, lakehouse: str, destination: str, projectedColumns: list = [], addId: bool = True) -> None:
    """
    Handling overwriting to Delta. Writes use the fully qualified
    '<lakehouse>.<schema>.<table>' path required by schema-enabled Fabric lakehouses.
    The schema and table are derived from destination = '<schema>.<table>'.

    Args:
        df (pyspark.sql.DataFrame): The DataFrame to write.
        lakehouse (str): Destination lakehouse name.
        destination (str): '<schema>.<table>' inside the lakehouse.
        projectedColumns (list): Columns to write. [] is all columns. [] is default.
        addId (bool): Adding a faux auto identity column if not exists. True is default.

    Returns:
        None
    """
    schema, table = _split_destination(destination)
    fq_name = f"{lakehouse}.{schema}.{table}"
    print(f"Overwriting {fq_name}")

    projectedColumns = projectedColumns if projectedColumns != [] else [column[0] for column in df.dtypes]

    existing_cols_lower = [column[0].lower() for column in df.dtypes]
    if "id" not in existing_cols_lower and addId:
        df = df.withColumn("Id", F.monotonically_increasing_id() + 1)
        projectedColumns.append('Id')
        df = df.select(projectedColumns)

    df.write.format('delta').option("mergeSchema", "true").mode('overwrite').saveAsTable(fq_name)

def load_dimension(df: DataFrame, lakehouse: str, destination: str) -> None:
    """
    Handling loading of dimensions.

    Args:
        df (pyspark.sql.DataFrame): The DataFrame to write.
        lakehouse (str): Destination lakehouse name.
        destination (str): '<schema>.<table>' inside the lakehouse.

    Returns:
        None
    """
    write_to_delta_overwrite(df, lakehouse, destination, [], False)

def load_fact(df: DataFrame, lakehouse: str, destination: str) -> None:
    """
    Handling loading of facts.

    Args:
        df (pyspark.sql.DataFrame): The DataFrame to write.
        lakehouse (str): Destination lakehouse name.
        destination (str): '<schema>.<table>' inside the lakehouse.

    Returns:
        None
    """
    write_to_delta_overwrite(df, lakehouse, destination, [], False)


## **Utility Functions**

#### Each function in this notebook, should make more easy to create a data platform. An another example of utility function is generators. In this case we have a generator to genereate a date table. 

In [ ]:
def create_or_update_date_dimension(start_date: str, end_date: str, lakehouse: str, destination: str) -> None:
    """
    Create or refresh a date dimension table.

    Args:
        start_date (str): From date.
        end_date (str): To date.
        lakehouse (str): Destination lakehouse name.
        destination (str): '<schema>.<table>' inside the lakehouse.

    Returns:
        None
    """
    df = spark.createDataFrame([(start_date, end_date)], ["start_date", "end_date"])
    df = df.selectExpr("explode(sequence(to_date(start_date), to_date(end_date))) as Date")
    df = df.withColumn("DateId", F.expr("cast(date_format(date, 'yyyyMMdd') as int)"))
    df = df.select(['DateId', 'Date'])

    df = df.withColumn("Year", F.year(F.col("Date")))
    df = df.withColumn("Month", F.month(F.col("Date")))
    df = df.withColumn("Day", F.dayofmonth(F.col("Date")))

    write_to_delta_overwrite(df=df, lakehouse=lakehouse, destination=destination)


## **Custom transform registry**

#### Registry primitives for the Bronze to Silver extension point. The registry itself lives here in the functions library because it is framework code. Actual transform implementations are registered from a separate notebook (`custom_transforms`) so user code and framework code stay apart.

#### Runtime contract: `functions` must be `%run` first, so `register_custom_transform` exists before `custom_transforms` uses it.


In [ ]:
CUSTOM_TRANSFORMS = {}

def register_custom_transform(name: str):
    """
    Register a custom Bronze to Silver transform under a string name.
    Reference it from meta_data.json via the custom_transform property.
    """
    def decorator(fn):
        CUSTOM_TRANSFORMS[name] = fn
        return fn
    return decorator

def run_custom_transform(name: str, df: DataFrame, meta_entity: dict) -> DataFrame:
    if name not in CUSTOM_TRANSFORMS:
        raise KeyError(f"Custom transform {name!r} is not registered. Known: {list(CUSTOM_TRANSFORMS)}")
    return CUSTOM_TRANSFORMS[name](df, meta_entity)
